In [ ]:
# Install if needed (uncomment if running first time)
# !pip install xgboost optuna imbalanced-learn --quiet

import os, time, joblib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, recall_score, precision_score,f1_score, confusion_matrix, roc_auc_score, average_precision_score)

from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler

# Create folders
os.makedirs("../models", exist_ok=True)
os.makedirs("../tuning", exist_ok=True)


In [ ]:
DATA_PATH = "creditcard.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print(df['Class'].value_counts())
df.head()


Shape: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


- Class = 1 is fraud (very rare).
- We will NOT use SMOTE in this pipeline; instead we'll use `scale_pos_weight` in XGBoost and CV.




In [ ]:
# FEATURES / LABEL
X = df.drop("Class", axis=1)
y = df["Class"]

# Train / Temp split (70/30) then val/test (half/half -> 15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# Scale Time & Amount
scaler = StandardScaler()
for col in ["Time", "Amount"]:
    X_train[col] = scaler.fit_transform(X_train[[col]])
    X_val[col]   = scaler.transform(X_val[[col]])
    X_test[col]  = scaler.transform(X_test[[col]])

# Persist scaler
joblib.dump(scaler, "../models/scaler.joblib")
print("Saved scaler at ../models/scaler.joblib")

Train: (199364, 30) Val: (42721, 30) Test: (42722, 30)
Saved scaler at ../models/scaler.joblib


In [ ]:
# compute scale_pos_weight from train
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

baseline_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "use_label_encoder": False,
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

# Fit on training (no SMOTE)
model_baseline = XGBClassifier(**baseline_params)
model_baseline.fit(X_train, y_train)

# Evaluation function
def evaluate_model(clf, X, y, label="validation"):
    proba = clf.predict_proba(X)[:,1]
    pred = (proba > 0.5).astype(int)
    rec = recall_score(y, pred)
    prec = precision_score(y, pred, zero_division=0)
    f1 = f1_score(y, pred, zero_division=0)
    auc = roc_auc_score(y, proba)
    cm = confusion_matrix(y, pred)
    print(f"\n=== {label.upper()} RESULTS ===")
    print(f"Recall: {rec:.4f}, Precision: {prec:.4f}, F1: {f1:.4f}, ROC-AUC: {auc:.4f}")
    print("Confusion matrix:\n", cm)
    return {"recall":rec, "precision":prec, "f1":f1, "auc":auc, "cm":cm}

print("Baseline on VAL")
baseline_val_metrics = evaluate_model(model_baseline, X_val, y_val, "val")
print("Baseline on TEST")
baseline_test_metrics = evaluate_model(model_baseline, X_test, y_test, "test")

# Save baseline model
joblib.dump(model_baseline, "../models/xgb_baseline.joblib")
print("Saved baseline model to ../models/xgb_baseline.joblib")


scale_pos_weight = 578.55
Baseline on VAL

=== VAL RESULTS ===
Recall: 0.7838, Precision: 0.9062, F1: 0.8406, ROC-AUC: 0.9721
Confusion matrix:
 [[42641     6]
 [   16    58]]
Baseline on TEST

=== TEST RESULTS ===
Recall: 0.7973, Precision: 0.8806, F1: 0.8369, ROC-AUC: 0.9552
Confusion matrix:
 [[42640     8]
 [   15    59]]
Saved baseline model to ../models/xgb_baseline.joblib


This Optuna section:
- 5-fold Stratified CV within the training data
- Optimize mean AUPRC (average precision)
- Enforce mean recall >= TARGET_RECALL and mean precision >= TARGET_PRECISION to save models
- Prevent mode collapse by rejecting trials that predict >90% fraud
- Use expanded hyperparameter space


In [ ]:
# TUNING CONFIG
TARGET_RECALL = 0.90
TARGET_PRECISION = 0.10
N_TRIALS = 200           # increased
MAX_TIME = 2 * 60 * 60   # 2 hours
START_TIME = time.time()

# Use the training split only for CV
X_train_bal = X_train.copy()
y_train_bal = y_train.copy()

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 400, 2500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.0005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        # scale_pos_weight: important to reflect class imbalance
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", max(1, scale_pos_weight*0.5), scale_pos_weight*2.0),
        "use_label_encoder": False,
        "eval_metric": "aucpr",
        "n_jobs": -1,
        "random_state": 42
    }

    # Cross-validated training and evaluation
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auprc_scores, recall_scores, precision_scores = [], [], []

    for tr_idx, val_idx in kf.split(X_train_bal, y_train_bal):
        X_tr, X_val_fold = X_train_bal.iloc[tr_idx], X_train_bal.iloc[val_idx]
        y_tr, y_val_fold = y_train_bal.iloc[tr_idx], y_train_bal.iloc[val_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)

        proba = model.predict_proba(X_val_fold)[:,1]
        preds = (proba > 0.5).astype(int)

        # Reject trial if degenerate (predicting almost all as fraud)
        if preds.sum() > 0.9 * len(preds):
            return 0.0

        auprc_scores.append(average_precision_score(y_val_fold, proba))
        recall_scores.append(recall_score(y_val_fold, preds))
        precision_scores.append(precision_score(y_val_fold, preds, zero_division=0))

        # time guard for long training inside kfold
        if time.time() - START_TIME > MAX_TIME:
            raise optuna.exceptions.OptunaError("TIME_LIMIT_REACHED")

    mean_auprc = np.mean(auprc_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)

    # Save model only if both constraints satisfied (save last fold model for simplicity)
    if (mean_recall >= TARGET_RECALL) and (mean_precision >= TARGET_PRECISION):
        print(f"\n🎉 Found candidate: AUPRC={mean_auprc:.4f}, Recall={mean_recall:.4f}, Precision={mean_precision:.4f}")
        # Fit final model on whole training set
        final_model = XGBClassifier(**params)
        final_model.fit(X_train_bal, y_train_bal)
        joblib.dump(final_model, "../models/xgb_optuna_cv_best.pkl")
        joblib.dump(params, "../models/xgb_optuna_cv_params.pkl")

    return mean_auprc

# Run study
sampler = TPESampler(seed=42)
study = optuna.create_study(direction="maximize", sampler=sampler)

try:
    study.optimize(objective, n_trials=N_TRIALS)
except optuna.exceptions.OptunaError as e:
    print("Optuna stopped early:", e)

print("Study complete. Best AUPRC:", study.best_value)
print("Best params:", study.best_params)


[I 2025-11-15 14:28:15,725] A new study created in memory with name: no-name-5d2b43e8-4fde-45de-bf3a-6008771eb5f9
[I 2025-11-15 14:29:28,115] Trial 0 finished with value: 0.8629548377349907 and parameters: {'n_estimators': 1186, 'max_depth': 15, 'learning_rate': 0.05402066039246068, 'subsample': 0.799[...]
[I 2025-11-15 14:30:10,401] Trial 1 finished with value: 0.8599699710404565 and parameters: {'n_estimators': 443, 'max_depth': 15, 'learning_rate': 0.1027120903202408, 'subsample': 0.60616[...]
[I 2025-11-15 14:32:13,833] Trial 2 finished with value: 0.7905261447546575 and parameters: {'n_estimators': 1685, 'max_depth': 4, 'learning_rate': 0.0032403507666874456, 'subsample': 0.68[...]
[I 2025-11-15 14:34:04,351] Trial 3 finished with value: 0.7408712217570296 and parameters: {'n_estimators': 1676, 'max_depth': 5, 'learning_rate': 0.0007580418253702252, 'subsample': 0.97[...]
[I 2025-11-15 14:34:59,985] Trial 4 finished with value: 0.7579674258302684 and parameters: {'n_estimators': 6


🎉 Found candidate: AUPRC=0.7531, Recall=0.9040, Precision=0.1118


[I 2025-11-15 15:10:46,298] Trial 29 finished with value: 0.7531089925826342 and parameters: {'n_estimators': 1852, 'max_depth': 3, 'learning_rate': 0.0018834373545101363, 'subsample': 0.6[...]
[I 2025-11-15 15:12:53,008] Trial 30 finished with value: 0.8553664413829285 and parameters: {'n_estimators': 1563, 'max_depth': 11, 'learning_rate': 0.024844016538746745, 'subsample': 0.5[...]
[I 2025-11-15 15:15:08,431] Trial 31 finished with value: 0.8593653642649123 and parameters: {'n_estimators': 1877, 'max_depth': 13, 'learning_rate': 0.03260198728511489, 'subsample': 0.72[...]
[I 2025-11-15 15:16:55,273] Trial 32 finished with value: 0.8626942398718874 and parameters: {'n_estimators': 1812, 'max_depth': 14, 'learning_rate': 0.0906385276838068, 'subsample': 0.725[...]
[I 2025-11-15 15:18:35,405] Trial 33 finished with value: 0.8557251622773056 and parameters: {'n_estimators': 1722, 'max_depth': 15, 'learning_rate': 0.0863766854301819, 'subsample': 0.662[...]
[I 2025-11-15 15:20:22,249] Tr

Optuna stopped early: TIME_LIMIT_REACHED
Study complete. Best AUPRC: 0.8645590671932532
Best params: {'n_estimators': 2136, 'max_depth': 10, 'learning_rate': 0.12222254620589187, 'subsample': 0.5617806456901816, 'colsample_bytree': 0.9537476840194699, 'gamma': 0.3064964140044[...]


In [15]:
best_model = joblib.load("../models/xgb_optuna_cv_best.pkl")
best_params = joblib.load("../models/xgb_optuna_cv_params.pkl")

best_model, best_params


(XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=0.7333043248949551, device=None,
               early_stopping_rounds=None, enable_categorical=False,
               eval_metric='aucpr', feature_types=None, feature_weights=None,
               gamma=0.8386597518027588, grow_policy=None, importance_type=None,
               interaction_constraints=None, learning_rate=0.0018834373545101363,
               max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
               max_delta_step=None, max_depth=3, max_leaves=None,
               min_child_weight=3, missing=nan, monotone_constraints=None,
               multi_strategy=None, n_estimators=1852, n_jobs=-1,
               num_parallel_tree=None, ...),
 {'n_estimators': 1852,
  'max_depth': 3,
  'learning_rate': 0.0018834373545101363,
  'subsample': 0.6776936071181332,
  'colsample_bytree': 0.7333043248949551,
  'gamma': 

In [16]:
val_proba = best_model.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.01, 0.50, 100)

best_thr = 0.5
best_f1 = 0

best_result = {"threshold":0.5, "recall":0, "precision":0, "f1":0}

for t in thresholds:
    preds = (val_proba > t).astype(int)
    rec = recall_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds)

    # prioritize meeting constraints
    if rec >= 0.90 and prec >= 0.10:
        best_result = {
            "threshold": t,
            "recall": rec,
            "precision": prec,
            "f1": f1
        }
        break

    # otherwise maximize F1
    if f1 > best_f1:
        best_f1 = f1
        best_thr = t
        best_result = {
            "threshold": t,
            "recall": rec,
            "precision": prec,
            "f1": f1
        }

best_result

{'threshold': 0.5,
 'recall': 0.8783783783783784,
 'precision': 0.09687034277198212,
 'f1': 0.174496644295302}

In [17]:
best_threshold = best_result["threshold"]

test_proba = best_model.predict_proba(X_test)[:, 1]
test_preds = (test_proba > best_threshold).astype(int)

test_recall = recall_score(y_test, test_preds)
test_precision = precision_score(y_test, test_preds, zero_division=0)
test_f1 = f1_score(y_test, test_preds)
test_auc = roc_auc_score(y_test, test_proba)
test_cm = confusion_matrix(y_test, test_preds)

print("\n===== FINAL TEST RESULTS =====")
print("Threshold:", best_threshold)
print(f"Recall:    {test_recall:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"F1-score:  {test_f1:.4f}")
print(f"AUC:       {test_auc:.4f}")
print("Confusion Matrix:\n", test_cm)

print("\nClassification Report:\n")
print(classification_report(y_test, test_preds, digits=4))



===== FINAL TEST RESULTS =====
Threshold: 0.5
Recall:    0.8649
Precision: 0.0936
F1-score:  0.1689
AUC:       0.9619
Confusion Matrix:
 [[42028   620]
 [   10    64]]

Classification Report:

              precision    recall  f1-score   support

           0     0.9998    0.9855    0.9926     42648
           1     0.0936    0.8649    0.1689        74

    accuracy                         0.9853     42722
   macro avg     0.5467    0.9252    0.5807     42722
weighted avg     0.9982    0.9853    0.9911     42722

